# 10. Нарезка ортофотомозаики на train / val / test

Из тайлов `Amsterdam` и `Field` (GeoTIFF, 8 см) нарезаются кропы 512×512 пикселей HQ.

| Выборка | Кропов | Amsterdam | Field |
| --- | --- | --- | --- |
| train | 7000 | 3500 | 3500 |
| val | 600 | 300 | 300 |
| test | 1000 | 500 | 500 |

**Как обеспечено отсутствие пересечений**
1. Каждый тайл целиком попадает **только в одну** выборку (разбиение по файлам, а не по кропам).
2. Кропы берутся не ближе `EDGE_MARGIN` пикселей к краю тайла, поэтому кропы из соседних тайлов разных выборок разделены зазором не меньше `2 × EDGE_MARGIN` (по умолчанию 1024 пикселя ≈ 82 м).
3. Внутри тайла кропы лежат на сетке с шагом `CROP`, то есть не перекрываются друг с другом.
4. Для val и test берётся много тайлов с небольшим числом кропов на тайл (`MAX_PER_TILE`), чтобы выборки были разнообразными.

Последняя ячейка проверяет всё перечисленное и строит карту тайлов.

## Инструкция
1. **Настройки ноутбука.** Accelerator — `None` (GPU не нужен), Internet — `On` (для `pip install rasterio`).
2. **Add Input** — датасеты с папками `Amsterdam` и `Field`.
3. Если папки найдены неверно, впишите пути в `AMS_DIR` и `FIELD_DIR`.
4. Запустите ячейки по порядку. Результат — `/kaggle/working/dataset/{train,val,test}/hq/*.png`, `manifest.csv` и `split_summary.json`.
5. **Сохранение.** `Save Version` → `Save & Run All`, затем из вывода (Output) создайте датасет (`New Dataset`).

In [ ]:
!pip install -q rasterio

In [ ]:
import os, glob, re, json, math, random, csv, shutil
import numpy as np
from PIL import Image
import rasterio
from rasterio.windows import Window

# ==== Пути ====
INPUT_ROOT = "/kaggle/input"
AMS_DIR    = None     # None = найти автоматически папку с названием Amsterdam
FIELD_DIR  = None     # None = найти автоматически папку с названием Field
OUT_DIR    = "/kaggle/working/dataset"

# ==== Параметры нарезки ====
CLASSES  = ["Amsterdam", "Field"]
CROP     = 512                                              # сторона кропа в пикселях HQ (кратна 4)
SIZES    = {"train": 7000, "val": 600, "test": 1000}        # поровну на каждый класс
MAX_PER_TILE = {"val": 60, "test": 60, "train": None}       # среднее число кропов на тайл в val и test (разнообразие)
EDGE_MARGIN  = 512                                          # отступ от края тайла: зазор между выборками
MIN_STD      = 4.0                                          # отбраковка почти однотонных кропов (яркость 0–255)
MAX_NODATA   = 0.02                                         # доля чисто чёрных/белых пикселей, выше — отбраковка
SEED     = 42
SAVE_COMPRESS = 3                                           # PNG без потерь; уровень 1–9 (больше = меньше файл, дольше)

In [ ]:
def find_tifs(explicit, name):
    roots = [explicit] if explicit else sorted(d for d, _, _ in os.walk(INPUT_ROOT) if os.path.basename(d).lower() == name.lower())
    files = sorted({f for r in roots for pat in ("*.tif", "*.tiff") for f in glob.glob(os.path.join(r, "**", pat), recursive=True)})
    return roots, files

DIRS = {"Amsterdam": AMS_DIR, "Field": FIELD_DIR}
TILES = {}
for c in CLASSES:
    roots, files = find_tifs(DIRS[c], c)
    print(f"{c}: папки {roots}, файлов {len(files)}")
    assert files, f"не найдены .tif для {c}: укажите путь вручную"
    TILES[c] = files
print("ожидалось: Amsterdam 59, Field 36")

In [ ]:
TILE_RE = re.compile(r"(\d{4})_(\d+)_(\d+)")
INFO = {}                                            # путь -> параметры тайла
for c in CLASSES:
    for p in TILES[c]:
        m = TILE_RE.search(os.path.basename(p))
        key = m.group(0) if m else os.path.splitext(os.path.basename(p))[0]
        with rasterio.open(p) as d:
            assert d.count >= 3, f"{p}: меньше 3 каналов"
            assert d.dtypes[0] == "uint8", f"{p}: тип {d.dtypes[0]}, ожидался uint8"
            INFO[p] = dict(cls=c, key=key, path=p, w=d.width, h=d.height,
                           x=int(m.group(2)) if m else 0, y=int(m.group(3)) if m else 0)
sizes = sorted({(i["w"], i["h"]) for i in INFO.values()})
print("размеры тайлов (ширина, высота):", sizes[:5], "..." if len(sizes) > 5 else "")
keys = [i["key"] for i in INFO.values()]
assert len(keys) == len(set(keys)), "есть тайлы с одинаковым ключом в имени (Amsterdam и Field не должны повторяться)"

## Распределение тайлов по выборкам

Тайлы каждого класса перемешиваются (`SEED`). Сначала берутся тайлы для val и test, остальные идут в train.
Число тайлов для val и test выбирается так, чтобы на тайл приходилось не больше `MAX_PER_TILE` кропов.

In [ ]:
for s, n in SIZES.items():
    assert n % len(CLASSES) == 0, f"{s}: {n} не делится поровну на классы"
    assert n // len(CLASSES) % 1 == 0 and CROP % 4 == 0

rng = random.Random(SEED)
ASSIGN = {}                                          # класс -> выборка -> список тайлов
for c in CLASSES:
    ts = sorted(TILES[c]); rng.shuffle(ts)
    k_val  = math.ceil(SIZES["val"]  / len(CLASSES) / MAX_PER_TILE["val"])
    k_test = math.ceil(SIZES["test"] / len(CLASSES) / MAX_PER_TILE["test"])
    assert len(ts) > k_val + k_test, f"{c}: мало тайлов ({len(ts)}) для val ({k_val}) и test ({k_test})"
    ASSIGN[c] = {"val": ts[:k_val], "test": ts[k_val:k_val + k_test], "train": ts[k_val + k_test:]}
    print(f"{c}: тайлов train {len(ASSIGN[c]['train'])}, val {k_val}, test {k_test}  "
          f"(кропов на тайл: train ~{SIZES['train'] / len(CLASSES) / len(ASSIGN[c]['train']):.0f}, "
          f"val ~{SIZES['val'] / len(CLASSES) / k_val:.0f}, test ~{SIZES['test'] / len(CLASSES) / k_test:.0f})")

## Нарезка

Кропы берутся по кругу из всех тайлов выборки (так выбор получается равномерным по тайлам) в случайных позициях сетки с шагом `CROP`.
Кропы с большим количеством пустых пикселей и почти однотонные отбраковываются. Файлы пишутся в PNG без потерь.

In [ ]:
def candidates(info):
    """Позиции левых верхних углов кропов: сетка с шагом CROP внутри отступа EDGE_MARGIN, случайно сдвинутая и перемешанная."""
    r = random.Random(f"{SEED}-{info['key']}")
    ox, oy = r.randrange(CROP), r.randrange(CROP)
    xs = range(EDGE_MARGIN + ox, info["w"] - EDGE_MARGIN - CROP + 1, CROP)
    ys = range(EDGE_MARGIN + oy, info["h"] - EDGE_MARGIN - CROP + 1, CROP)
    cells_ = [(x, y) for y in ys for x in xs]
    r.shuffle(cells_)
    return cells_

def is_valid(arr):
    nod = ((arr.max(0) == 0) | (arr.min(0) == 255)).mean()
    return nod <= MAX_NODATA and arr.astype(np.float32).mean(0).std() >= MIN_STD

def cut_split(split, rows):
    for c in CLASSES:
        n = SIZES[split] // len(CLASSES)
        ts = [INFO[p] for p in ASSIGN[c][split]]
        outdir = os.path.join(OUT_DIR, split, "hq"); os.makedirs(outdir, exist_ok=True)
        ds = [rasterio.open(t["path"]) for t in ts]
        cand = [candidates(t) for t in ts]
        ptr = [0] * len(ts); taken = [0] * len(ts); got = 0
        try:
            while got < n:
                progressed = False
                for i, t in enumerate(ts):
                    if got >= n: break
                    while ptr[i] < len(cand[i]):
                        x, y = cand[i][ptr[i]]; ptr[i] += 1
                        arr = ds[i].read([1, 2, 3], window=Window(x, y, CROP, CROP))
                        if not is_valid(arr): continue
                        name = f"{c}_{t['key']}_x{x}_y{y}.png"
                        Image.fromarray(arr.transpose(1, 2, 0)).save(os.path.join(outdir, name), compress_level=SAVE_COMPRESS)
                        rows.append(dict(split=split, cls=c, tile=t["key"], file=name, x=x, y=y))
                        taken[i] += 1; got += 1; progressed = True
                        break
                if not progressed:
                    raise RuntimeError(f"{split}/{c}: тайлы исчерпаны на {got} из {n} кропов. "
                                       f"Уменьшите SIZES, EDGE_MARGIN или MIN_STD, либо увеличьте число тайлов.")
        finally:
            for d in ds: d.close()
        print(f"{split:5s} {c:10s} {got} кропов из {len(ts)} тайлов (на тайл: от {min(taken)} до {max(taken)})")

shutil.rmtree(OUT_DIR, ignore_errors=True)
ROWS = []
for split in ("test", "val", "train"):               # от меньших к большим: ошибки видны раньше
    cut_split(split, ROWS)
with open(os.path.join(OUT_DIR, "manifest.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(ROWS[0].keys())); w.writeheader(); w.writerows(ROWS)
print("всего кропов:", len(ROWS), "-> manifest.csv")

## Проверка

Проверяется: точное число кропов и 50/50 по классам в каждой выборке; тайлы не повторяются в разных выборках;
кропы внутри тайла не перекрываются и лежат не ближе `EDGE_MARGIN` к краю; файлы читаются и имеют размер `CROP`×`CROP`.

In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(OUT_DIR, "manifest.csv"))

# 1. размеры выборок и баланс классов
tab = df.groupby(["split", "cls"]).size().unstack(fill_value=0).loc[list(SIZES)]
display(tab)
for s, n in SIZES.items():
    assert len(df[df.split == s]) == n, f"{s}: {len(df[df.split == s])} вместо {n}"
    assert all(tab.loc[s, c] == n // len(CLASSES) for c in CLASSES), f"{s}: классы не поровну"

# 2. тайлы не пересекаются между выборками
sets = {s: set(df[df.split == s].tile) for s in SIZES}
for a in SIZES:
    for b in SIZES:
        if a < b: assert not (sets[a] & sets[b]), f"тайлы {a} и {b} пересекаются: {sets[a] & sets[b]}"

# 3. внутри тайла: нет перекрытий и выдержан отступ от края
for (_, t), g in df.groupby(["split", "tile"]):
    info = next(i for i in INFO.values() if i["key"] == t)
    assert (g.x >= EDGE_MARGIN).all() and (g.y >= EDGE_MARGIN).all()
    assert (g.x + CROP <= info["w"] - EDGE_MARGIN).all() and (g.y + CROP <= info["h"] - EDGE_MARGIN).all()
    xs, ys = g.x.values, g.y.values
    for i in range(len(g)):
        ov = (np.abs(xs[i] - xs) < CROP) & (np.abs(ys[i] - ys) < CROP); ov[i] = False
        assert not ov.any(), f"перекрытие кропов в тайле {t}"

# 4. файлы читаются и имеют нужный размер, имена не повторяются
assert df.file.is_unique
for s in SIZES:
    for fn in df[df.split == s].file.sample(min(50, SIZES[s]), random_state=0):
        assert Image.open(os.path.join(OUT_DIR, s, "hq", fn)).size == (CROP, CROP)
print("проверка пройдена")

summary = {"seed": SEED, "crop": CROP, "sizes": SIZES, "edge_margin": EDGE_MARGIN,
           "tiles": {s: sorted(sets[s]) for s in SIZES}}
json.dump(summary, open(os.path.join(OUT_DIR, "split_summary.json"), "w"), ensure_ascii=False, indent=1)

In [ ]:
# Карта тайлов: координаты из имён файлов (RD, метры). Цвет — выборка, форма — класс.
import matplotlib.pyplot as plt
colors = {"train": "#4c78a8", "val": "#f58518", "test": "#e45756"}
fig, ax = plt.subplots(figsize=(9, 7))
for info in INFO.values():
    s = next(s for s in SIZES if info["key"] in sets[s])
    ax.scatter(info["x"], info["y"], c=colors[s], marker="s" if info["cls"] == "Amsterdam" else "^", s=60, label=f"{s} / {info['cls']}")
h, l = ax.get_legend_handles_labels(); u = dict(zip(l, h)); ax.legend(u.values(), u.keys(), fontsize=8)
ax.set_xlabel("x, м (RD)"); ax.set_ylabel("y, м (RD)"); ax.grid(alpha=.3); ax.set_title("Тайлы по выборкам (квадрат — Amsterdam, треугольник — Field)")
plt.show()

## Что делать дальше

- В `PATHS` ноутбуков обучения укажите `TRAIN_HQ = ".../dataset/train/hq"`, `VAL_HQ = ".../dataset/val/hq"`, а тестовую выборку (`dataset/test/hq`) используйте один раз, для итоговых таблиц.
- LQ создаются в обучающих ноутбуках ячейкой `make_lq` (для test: `--seed 2`, чтобы деградация отличалась от val).
- После новой нарезки перегоните диагностику `00` на новом val.